In [1]:
import pandas as pd
import joblib
import scipy.sparse as sp
import numpy as np

ridge = joblib.load("../models/ridge_baseline.joblib")
print("Ridge model loaded successfully, alpha =", ridge.alpha)

Ridge model loaded successfully, alpha = 5.0


In [2]:
# Load remaining saved artifacts
name_vec = joblib.load("../models/name_vectorizer.joblib")
desc_vec = joblib.load("../models/desc_vectorizer.joblib")
scaler   = joblib.load("../models/scaler.joblib")
ohe      = joblib.load("../models/onehot_encoder.joblib")

# Load validation data
val = pd.read_csv("../data/processed/val.csv")
val["name"] = val["name"].fillna("")
val["item_description"] = val["item_description"].fillna("")

NUM_COLS = ["item_condition_id","shipping","category_depth","name_length","desc_length",
            "name_word_count","has_description","is_branded","cat_avg_price","brand_avg_price"]
CAT_COLS = ["main_category","sub_category","sub_sub_category","condition_label"]
for c in CAT_COLS:
    val[c] = val[c].fillna("missing")

# Transform only -- these were already fitted on train in train_ridge.py
X_name_val = name_vec.transform(val["name"])
X_desc_val = desc_vec.transform(val["item_description"])
X_num_val = scaler.transform(val[NUM_COLS])
X_cat_val = ohe.transform(val[CAT_COLS])

X_val = sp.hstack([X_name_val, X_desc_val, sp.csr_matrix(X_num_val), X_cat_val]).tocsr()


print("X_val shape:", X_val.shape)

X_val shape: (9945, 25759)


In [3]:
ridge.predict(X_val)

array([3.10573566, 2.88928125, 2.21488466, ..., 2.51781581, 3.52460409,
       2.59732849], shape=(9945,))

In [4]:
pred_log = ridge.predict(X_val)
y_val = val["log_price"].values
pred_price = np.expm1(pred_log)
actual_price = np.expm1(y_val)

abs_error = np.abs(actual_price - pred_price)

print("Mean absolute error ($):", round(abs_error.mean(), 2))
print("Median absolute error ($):", round(np.median(abs_error), 2))
print("90th percentile error ($):", round(np.percentile(abs_error, 90), 2))
print("99th percentile error ($):", round(np.percentile(abs_error, 99), 2))
print("Max error ($):", round(abs_error.max(), 2))

Mean absolute error ($): 11.9
Median absolute error ($): 5.3
90th percentile error ($): 22.99
99th percentile error ($): 115.1
Max error ($): 1450.12


In [5]:
val_results = val.copy()
val_results["predicted_price"] = pred_price
val_results["actual_price"] = actual_price
val_results["abs_error"] = abs_error

worst = val_results.sort_values("abs_error", ascending=False).head(10)
worst[["name", "main_category", "brand_name", "actual_price", "predicted_price", "abs_error"]]

,name,main_category,brand_name,actual_price,predicted_price,abs_error
8567,Chanel Classic Flag Bag medium Caviar L,Women,chanel,1506.0,55.884074,1450.115926
6567,Alexander McQueen Crystal Knuckle Clutch,Vintage & Collectibles,alexander mcqueen,1109.0,35.489712,1073.510288
8982,Ricky Garner,Other,no brand,650.0,12.573833,637.426167
7326,Iphone 7 Plus 32gb Verizon Flawless,Electronics,apple,656.0,127.490330,528.509670
1267,Christian louboutin Tudor bal 100,Women,christian louboutin,589.0,68.657634,520.342366
4607,PlayStation VR,Electronics,sony,500.0,12.972919,487.027081
8431,Lularoe LARGE Joy,Women,no brand,525.0,50.980761,474.019239
6206,Favorite PM by Louis Vuitton,Women,louis vuitton,680.0,278.615922,401.384078
7333,14k bundle,Handmade,no brand,435.0,36.695249,398.304751
7319,Michael Kors watch bundle JoJo Boutique,Women,michael kors,450.0,53.463782,396.536218


In [6]:
val_results["signed_error"] = val_results["predicted_price"] - val_results["actual_price"]

price_tier_check = val_results.groupby("price_bin")["signed_error"].mean()
print(price_tier_check)

price_bin
0     5.207851
1     4.599350
2     3.656053
3     3.089727
4     1.864257
5     0.389693
6    -0.952890
7    -3.935433
8   -10.024551
9   -56.482281
Name: signed_error, dtype: float64
